# Config

In [ ]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [ ]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [30]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Interdisciplinario"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Interdisciplinario"]!="INDEFINIDO"]

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Interdisciplinario"])
savepath = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 193
Fold 0 - Val size: 257
Archivo guardado exitosamente en /tmp/final_project/dataSplits/interdiciplinario/train_test_ids_3folds.json


# 2) Entrenamiento TF-IDF

### Carga y entrenamiento

In [3]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [5]:
#################################### ESTO ES PARA TESTEAR DATOS ANTIGUOS #########################
#Ruta de lectura
path2 = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path2, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
#filepath=os.path.join(path2, "data_translated_concat.csv")
#df = pd.read_csv(filepath)


In [4]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdisciplinario")
sample = X_test[0]
#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdisciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
y_train = [1 if x == "SI" else 0 for x in y_train]
y_test = [1 if x == "SI" else 0 for x in y_test]

#Creacion de vectores TFID
X_train, X_test, vectorizer = gen_TFID_vectors(X_train, X_test, return_vectorizer=True)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


(771, 16515) (193, 16515)


In [10]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
import warnings
warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)


# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
split_idx_path = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
#split_idx_path = os.path.join(path2, "train_test_ids_3folds.json")
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.66, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.66, 'std_test_score': 0.03}


In [11]:
from utils.mlflow import eval_model

# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds = eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6217616580310881, 'f1_macro': 0.6110697032436163, 'cm': array([[44, 38],
       [35, 76]]), 'precision': 0.6666666666666666, 'recall': 0.6846846846846847, 'f1_es': 0.5779563725650845, 'f1_en': 0.6552552552552553, 'cm_es': array([[13, 28],
       [17, 53]]), 'cm_en': array([[31, 10],
       [18, 23]])}
RandomForestClassifier
{'accuracy': 0.616580310880829, 'f1_macro': 0.5965536723163842, 'cm': array([[38, 44],
       [30, 81]]), 'precision': 0.648, 'recall': 0.7297297297297297, 'f1_es': 0.5846947082259549, 'f1_en': 0.5926539214210447, 'cm_es': array([[ 9, 32],
       [ 9, 61]]), 'cm_en': array([[29, 12],
       [21, 20]])}
XGBClassifier
{'accuracy': 0.5699481865284974, 'f1_macro': 0.5544460823853364, 'cm': array([[37, 45],
       [38, 73]]), 'precision': 0.6186440677966102, 'recall': 0.6576576576576577, 'f1_es': 0.5410244418896185, 'f1_en': 0.5975011155734047, 'cm_es': array([[13, 28],
       [22, 48]]), 'cm_en': array([[24, 17],
       [16, 25]])}
SVC


### Guardado de resultados

In [12]:
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline

savepath = os.path.join(path, "output/interdiciplinario/TF_IDF")
models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results, preds, save_preds=True, mode_classification="binary")

📝 Registrando modelo: LogisticRegression
📝 Registrando modelo: RandomForestClassifier


/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:167: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [15]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/interdiciplinario/TF_IDF/models"
model_path = os.path.join(model_path, "SVC.joblib")

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Prediction: [1]


# 3) SPECTER

### Train

In [9]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [37]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdisciplinario")

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdisciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
y_train = [1 if x == "SI" else 0 for x in y_train]
y_test = [1 if x == "SI" else 0 for x in y_test]

# Calcular embeddings
# Parámetros para cargar modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [38]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'RandomForestClassifier',
    #'XGBClassifier',
    #'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
split_idx_path = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.02}


In [39]:
from utils.save_results import eval_model

# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds = eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.5906735751295337, 'f1_macro': 0.5818730289318526, 'cm': array([[43, 39],
       [40, 71]]), 'precision': 0.6454545454545455, 'recall': 0.6396396396396397, 'f1_es': 0.5226654195652033, 'f1_en': 0.6683146067415731, 'cm_es': array([[12, 29],
       [23, 47]]), 'cm_en': array([[31, 10],
       [17, 24]])}


### Guardado de resultados

In [ ]:
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline
from models.specter import BERT_vectorizer

savepath = os.path.join(path, "output/interdiciplinario/SPECTER")
vectorizer = BERT_vectorizer(BASE_MODEL, ADAPTER_NAME)
models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
save_models_and_metrics(savepath, results_val, models_dicc_pipeline, df_test, y_test, results, preds, save_preds=True, mode_classification="binary")

📝 Registrando modelo: LogisticRegression


### Inference

In [41]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/interdiciplinario/SPECTER/models"
model_path = os.path.join(model_path, "LogisticRegression.joblib")

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
Prediction: [1]
